In [26]:
import xarray as xr
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.ndimage import uniform_filter1d
from minisom import MiniSom
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pandas as pd
from sklearn.metrics import silhouette_score
import matplotlib.gridspec as gridspec
from collections import defaultdict
import cdsapi
from pathlib import Path

In [2]:
missing_dates = [
    '200204250000', '200208300000', '200304150000', '200304160000',
    '200306250000', '200307270000', '200307280000', '200312280000',
    '200404140000', '200408090000', '200905280000', '201105210000',
    '202005240000', '200510240000'
]
missing_dt = pd.to_datetime(missing_dates, format='%Y%m%d%H%M')
# selection of desired days
pph = xr.open_dataset('data/raw_data/labelled_pph.nc')
pph_time = pd.to_datetime(pph['time'].values, format='%Y%m%d%H%M')

cats = ['SLGT', 'ENH', 'MDT', 'HIGH']
mask = np.isin(pph['MAX_CAT'].values, cats) & (pph_time> '2002-04-01') & (pph_time < '2020-01-01') & ~np.isin(pph_time, missing_dt)

times = pph_time[mask]
times_shifted = times + np.timedelta64(1, 'D')

In [ ]:
grid_outlooks = xr.open_dataset('~/Downloads/grid_outlooks.nc')
grid_outlooks = grid_outlooks.sel(time = grid_outlooks['time'] >= '200203300000')
# finding x and y to make center for each date
grouped = grid_outlooks['prob'].groupby('time')

# Step 1: Find all points with the maximum prob for each day and compute mean coordinates
def find_mean_coords(group):
    max_prob = group.max()  # Maximum value in the group
    if max_prob == 0:
        mean_x = group['x'].mean().item()
        mean_y = group['y'].mean().item()
    else:
        # Select all points with prob == max_prob
        max_points = group.where(group == max_prob, drop=True)
        print(max_points)
        # Compute the mean of x and y
        mean_x = max_points['x'].mean().item()
        mean_y = max_points['y'].mean().item()
    return xr.Dataset({'nearest_x': mean_x, 'nearest_y': mean_y})

# Apply the function to each group
center_coords = grouped.map(find_mean_coords)
center_coords

In [19]:
# pph time is 'YYYYMMDDHHMM' strings
pph = pph.assign_coords(
    time=pd.to_datetime(pph.time.values, format="%Y%m%d%H%M")
)

center_coords = center_coords.assign_coords(
    time=pd.to_datetime(center_coords.time.values, format="%Y%m%d%H%M")
)

cc = center_coords.sel(time=times)

latlon = xr.Dataset(
    {
        "lat": pph.lat,
        "lon": pph.lon,
    }
)

interp_latlon = latlon.interp(
    x=xr.DataArray(cc.nearest_x, dims="time"),
    y=xr.DataArray(cc.nearest_y, dims="time"),
    method="linear"
)

In [21]:
interp_latlon['time'] = times_shifted

In [ ]:
lats = interp_latlon.lat.values
lons = interp_latlon.lon.values

pad = 12.0

north = np.max(lats) + pad
south = np.min(lats) - pad
east  = np.max(lons) + pad
west  = np.min(lons) - pad

60.59944725036621 -132.3952751159668 13.247135639190674 -56.696462631225586


In [27]:
times = pd.to_datetime(interp_latlon.time.values)

req = defaultdict(lambda: defaultdict(list))

for t in times:
    req[t.year][t.month].append(f"{t.day:02d}")

In [30]:
c = cdsapi.Client()
DATA_DIR = Path("era5_conus")
DATA_DIR.mkdir(exist_ok=True)

def download_month(year, month, days, variables, levels, tag):
    fn = DATA_DIR / f"{tag}_{year}_{month:02d}.nc"
    print(f"Downloading {fn}...")
    if fn.exists():
        return

    c.retrieve(
        "reanalysis-era5-pressure-levels",
        {
            "product_type": "reanalysis",
            "format": "netcdf",
            "variable": variables,
            "pressure_level": levels,
            "year": str(year),
            "month": f"{month:02d}",
            "day": days,
            "time": "00:00",
            "area": [north, west, south, east]
        },
        str(fn),
    )

In [31]:
for y in req:
    for m in req[y]:
        download_month(
            y, m, req[y][m],
            variables=["geopotential"],
            levels=["500"],
            tag="z500"
        )

2026-04-28 15:45:57,077 INFO Request ID is 7e400e02-8b32-4dc3-9ff0-e12edbfce1cb
2026-04-28 15:45:57,270 INFO status has been updated to accepted
2026-04-28 15:46:19,253 INFO status has been updated to successful


4ee4fe65284f57ac2ebcc7940794dccb.nc:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

2026-04-28 15:46:23,030 INFO Request ID is c3caccfc-10e5-4f0e-948e-8acf6c4b5df1
2026-04-28 15:46:23,223 INFO status has been updated to accepted
2026-04-28 15:46:37,403 INFO status has been updated to running
2026-04-28 15:46:56,845 INFO status has been updated to successful


7cccfaa5e1221529ea1bcfe40e9e2cf1.nc:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

2026-04-28 15:46:59,954 INFO Request ID is 9eda641d-a81e-4ddd-ae37-bb824e61bf92
2026-04-28 15:47:00,145 INFO status has been updated to accepted
2026-04-28 15:47:14,249 INFO status has been updated to running
2026-04-28 15:47:33,815 INFO status has been updated to successful


53a1750e0751cdc73052d1e5aa357382.nc:   0%|          | 0.00/2.20M [00:00<?, ?B/s]

2026-04-28 15:47:37,020 INFO Request ID is ff332219-4569-4fa7-85fb-f34505c52f15
2026-04-28 15:47:37,217 INFO status has been updated to accepted
2026-04-28 15:47:51,337 INFO status has been updated to running


KeyboardInterrupt: 